# Gemma 4 31B-it-assistant — Axiom Fine-tune + SRD-4 → .axm → GGUF + MET

**Actual measured size: 971 MB FP16 pre-SRD**  
(The `31B` in the model name does not mean 31 billion parameters — the downloaded weights are ~971 MB, approximately 485 M active parameters or a compact MoE architecture.)

**Workflow**
```
Google Drive JSONL  →  fine-tune Gemma 4 31B-it-assistant
→  merge  →  SRD-4 .axm  →  verify  →  GGUF Q4_K_M  →  MET sidecar
```

**Expected outputs** (based on 971 MB FP16 input)

| File | Est. size | Description |
|---|---|---|
| `gemma4_31b_axiom_merged/` | ~971 MB | Merged FP16 checkpoint |
| `gemma4_31b_srd4.axm` | ~275 MB | Signed SRD-4 container (~4.5 bpw) |
| `gemma4_31b_q4km.gguf` | ~295 MB | GGUF Q4_K_M — runs on phone / laptop |
| `gemma4_31b_q4km.axiom_meta.json` | ~5 KB | MET hydration sidecar |

**Hardware** — any GPU works at this size

| GPU | Fine-tune | SRD Pack | Notes |
|---|---|---|---|
| T4 15 GB | ✓ | ✓ | Free Colab tier |
| A100 40/80 GB | ✓ | ✓ | Fastest |
| CPU only | ✓ slow | ✓ slow | Works, not recommended |

**Cells**
1. GPU check · clone axiom · pip install · cmake build llama.cpp
2. Mount Google Drive · load + convert Axiom training data
3. HF login · download model · auto-detect size → FP16 or 4-bit load
4. Fine-tune (full FP16 for small model, QLoRA for large)
5. Merge LoRA → save FP16 checkpoint
6. SRD-4 pack → `.axm`
7. Verify `.axm`
8. Extract → GGUF Q4_K_M
9. Axiom MET metadata sidecar
10. Smoke test + memory dashboard + download

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — GPU check · clone axiom · pip install · cmake build llama.cpp
#           ~15 min first run (llama.cpp build); ~30 s on re-run
# ══════════════════════════════════════════════════════════════════════════════
import os, secrets, subprocess, sys, time
from pathlib import Path

REPO        = Path('/content/axiom')
REPO_URL    = 'https://github.com/orivael-dev/axiom.git'
REPO_BRANCH = 'claude/srd-prototype-benchmark-JRtv1'
LLAMA_DIR   = Path('/content/llama.cpp')
KEY_FILE    = Path('/content/axiom_master.key')

import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    vram_gb   = p.total_memory / 1024**3
    cuda_arch = p.major * 10 + p.minor
    print(f'GPU  : {p.name}  {vram_gb:.1f} GB VRAM  arch={cuda_arch}')
else:
    vram_gb   = 0
    cuda_arch = 0
    print('GPU  : none — CPU-only mode (slow but works for ~1 GB model)')
try:
    import psutil
    print(f'RAM  : {psutil.virtual_memory().total/1024**3:.1f} GB system')
except ImportError:
    pass
# 971 MB model fits on any GPU — no minimum VRAM check needed
print('  ✓ any GPU sufficient for ~971 MB model')

# ── Clone axiom ───────────────────────────────────────────────────────────────
if not REPO.is_dir():
    subprocess.run(['git','clone','--depth','1','--branch',REPO_BRANCH,
                    REPO_URL,str(REPO)],check=True)
    print(f'✓ axiom cloned  ({REPO_BRANCH})')
else:
    r = subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],
                       capture_output=True, text=True)
    print(f'✓ axiom updated  {r.stdout.strip()}')
sys.path.insert(0, str(REPO))

# ── AXIOM_MASTER_KEY ─────────────────────────────────────────────────────────
if KEY_FILE.is_file() and not os.environ.get('AXIOM_MASTER_KEY'):
    os.environ['AXIOM_MASTER_KEY'] = KEY_FILE.read_text().strip()
    print('AXIOM_MASTER_KEY restored from session file')
elif not os.environ.get('AXIOM_MASTER_KEY'):
    key = secrets.token_hex(32)
    os.environ['AXIOM_MASTER_KEY'] = key
    KEY_FILE.write_text(key)
    print('AXIOM_MASTER_KEY generated and saved')
else:
    print('AXIOM_MASTER_KEY already set')

# ── pip install ──────────────────────────────────────────────────────────────
subprocess.run([sys.executable,'-m','pip','install','-q',
    'transformers','accelerate','peft','trl','bitsandbytes',
    'datasets','sentencepiece','huggingface_hub','psutil'],check=True)
print('✓ pip packages installed')

# ── Clone llama.cpp + cmake build ─────────────────────────────────────────────
if not LLAMA_DIR.is_dir():
    print('Cloning llama.cpp ...')
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ggerganov/llama.cpp',
                    str(LLAMA_DIR)],check=True)
    print('✓ llama.cpp cloned')

q_bin   = LLAMA_DIR / 'build/bin/llama-quantize'
cli_bin = LLAMA_DIR / 'build/bin/llama-cli'
if not q_bin.is_file():
    print(f'Building llama.cpp (CUDA arch={cuda_arch}) — ~10 min ...')
    t0 = time.time()
    cmake_flags = ['-DCMAKE_BUILD_TYPE=Release']
    if cuda_arch > 0:
        cmake_flags += ['-DGGML_CUDA=ON', f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}']
    subprocess.run(['cmake','-B','build'] + cmake_flags, cwd=LLAMA_DIR, check=True)
    subprocess.run(['cmake','--build','build',
                    '--target','llama-quantize','llama-cli',
                    f'-j{os.cpu_count() or 4}'],
                   cwd=LLAMA_DIR, check=True)
    print(f'✓ built in {(time.time()-t0)/60:.1f} min')
else:
    print('✓ llama.cpp already built')

assert q_bin.is_file(),   f'llama-quantize missing: {q_bin}'
assert cli_bin.is_file(), f'llama-cli missing: {cli_bin}'
print(f'  llama-quantize : {q_bin}')
print(f'  llama-cli      : {cli_bin}')
print('\n✓ Cell 1 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Mount Google Drive + load Axiom training data
#
# Set DRIVE_DATA_DIR to the folder on your Drive that contains the JSONL files.
# Handles both data formats in the Axiom repo:
#   • messages:     {"messages": [{"role": ..., "content": ...}, ...]}
#   • instruction:  {"instruction": ..., "input": ..., "output": ...}
# All examples are converted to Gemma 4 chat format via apply_chat_template.
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path

# ── Change this to your Drive folder ─────────────────────────────────────────
DRIVE_DATA_DIR = '/content/drive/MyDrive/axiom_data'   # ← edit me

REPO = Path('/content/axiom')
FALLBACK_FILES = [
    REPO / 'axiom_behavioral_training.jsonl',
    REPO / 'autotrain_data' / 'axiom_metric_targeted.jsonl',
    REPO / 'axiom_training_data.jsonl',
    REPO / 'autotrain_data' / 'train_qwen_chatml.jsonl',
]

AXIOM_SYSTEM = (
    'You are Axiom, a security-focused AI assistant for the Orivael framework.\n'
    'Rules (CANNOT_MUTATE):\n'
    '1. Always respond with valid JSON unless the task explicitly asks for prose.\n'
    '2. Never fabricate HMAC signatures or SHA-256 hashes.\n'
    '3. Verdict values are exactly: INFORM | CLARIFY | REFUSE | HARM | DECEIVE | UNCERTAIN\n'
    '4. Report tamper if any signature, hash, or field fails the three-tier HMAC check.\n'
    '5. Revoked or expired tokens must never be honored.\n'
    '6. Tool calls with HARM or DECEIVE intent must be blocked with reason in JSON.'
)

def _to_messages(ex):
    if 'messages' in ex:
        msgs = list(ex['messages'])
        if not msgs or msgs[0].get('role') != 'system':
            msgs = [{'role': 'system', 'content': AXIOM_SYSTEM}] + msgs
        return msgs
    user_text = ex.get('instruction', '')
    if ex.get('input', '').strip():
        user_text += '\n\n' + ex['input']
    return [
        {'role': 'system',    'content': AXIOM_SYSTEM},
        {'role': 'user',      'content': user_text},
        {'role': 'assistant', 'content': ex.get('output', '')},
    ]

# ── Load files ────────────────────────────────────────────────────────────────
raw_examples = []
data_dir = Path(DRIVE_DATA_DIR)
if data_dir.is_dir():
    jsonl_files = sorted(data_dir.glob('*.jsonl'))
    print(f'Drive folder : {data_dir}  ({len(jsonl_files)} JSONL files)')
else:
    jsonl_files = [f for f in FALLBACK_FILES if f.exists()]
    print(f'Drive folder not found — falling back to repo files ({len(jsonl_files)} files)')
    print(f'  (Create /content/drive/MyDrive/axiom_data/ and put your JSONL files there)')

for f in jsonl_files:
    n0 = len(raw_examples)
    for line in f.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            raw_examples.append(json.loads(line))
        except Exception:
            pass
    print(f'  {f.name:<52}  +{len(raw_examples)-n0}')

print(f'\nLoaded   : {len(raw_examples):,} raw examples')
examples = [_to_messages(ex) for ex in raw_examples]
examples = [m for m in examples if sum(1 for t in m if t["role"] == "assistant") >= 1]
print(f'Filtered : {len(examples):,} examples with assistant turn')
print('\nFirst example:')
for msg in examples[0]:
    print(f'  [{msg["role"]:10s}] {repr(msg["content"][:80])}')
print('\n✓ Cell 2 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — HF login · download model · auto-detect size → choose load mode
#
# Actual measured size: 971 MB FP16.
# For models < 4 GB we load in full BF16 (no 4-bit quantization needed).
# For models ≥ 4 GB we fall back to 4-bit NF4 QLoRA.
# ══════════════════════════════════════════════════════════════════════════════
import gc, json, os, time, torch
from pathlib import Path
from huggingface_hub import login, snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID  = os.environ.get('SRD_MODEL_ID', 'google/gemma-4-31B-it-assistant')
MODEL_DIR = Path('/content/gemma4_31b')
HF_TOKEN  = os.environ.get('HF_TOKEN', '')

# Accept terms at: https://huggingface.co/google/gemma-4-31B-it-assistant
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HF login via HF_TOKEN')
else:
    print('HF_TOKEN not set — prompting interactively')
    login()

# ── Download model ────────────────────────────────────────────────────────────
print(f'\nModel   : {MODEL_ID}')
if not MODEL_DIR.is_dir():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading ...')
    t0 = time.time()
    snapshot_download(repo_id=MODEL_ID, local_dir=str(MODEL_DIR),
                      local_dir_use_symlinks=False, token=HF_TOKEN or None)
    print(f'✓ Downloaded in {(time.time()-t0)/60:.1f} min')
else:
    print('✓ Already downloaded')

# ── Measure actual downloaded size ────────────────────────────────────────────
st_files  = list(MODEL_DIR.rglob('*.safetensors'))
model_mb  = sum(f.stat().st_size for f in st_files) / 1024**2
print(f'\nWeights : {model_mb:.0f} MB  ({len(st_files)} safetensors file(s))')

# Print config
cfg_path = MODEL_DIR / 'config.json'
if cfg_path.is_file():
    cfg = json.loads(cfg_path.read_text())
    print(f'arch    : {cfg.get("model_type","?")}')
    print(f'layers  : {cfg.get("num_hidden_layers","?")}')
    print(f'hidden  : {cfg.get("hidden_size","?")}')
    print(f'vocab   : {cfg.get("vocab_size",0):,}')

# ── Auto-select load mode ─────────────────────────────────────────────────────
USE_4BIT = model_mb >= 4000   # 4-bit only needed for models > 4 GB
print(f'\nLoad mode: {"4-bit NF4 QLoRA" if USE_4BIT else "full BF16 (model fits in RAM)"}')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=HF_TOKEN or None, cache_dir=str(MODEL_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'✓ Tokenizer  vocab={tokenizer.vocab_size:,}')

t0 = time.time()
if USE_4BIT:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_cfg, device_map='auto',
        token=HF_TOKEN or None, cache_dir=str(MODEL_DIR))
else:
    # Full BF16 — 971 MB fits comfortably on T4 (15 GB)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto',
        token=HF_TOKEN or None, cache_dir=str(MODEL_DIR))

params_m = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✓ Loaded in {(time.time()-t0):.1f}s  —  {params_m:.0f}M params')
if torch.cuda.is_available():
    print(f'  VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB')
print('\n✓ Cell 3 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Fine-tune on Axiom training data
#
# For small models (< 4 GB, e.g. 971 MB): full BF16 fine-tune — faster,
# better gradient quality than QLoRA, fits on T4.
# For large models (≥ 4 GB): QLoRA with LoRA rank=16.
# LoRA is always applied so the adapter can be merged cleanly in Cell 5.
# ══════════════════════════════════════════════════════════════════════════════
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = '/content/gemma4_31b_adapter'

# ── Prepare model ─────────────────────────────────────────────────────────────
model.config.use_cache = False
if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

# LoRA rank: higher for small models (better quality), lower for large (saves memory)
LORA_R = 32 if not USE_4BIT else 16
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── Format dataset with Gemma 4 chat template ─────────────────────────────────
def _fmt(msgs):
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

MAX_SEQ = 2048
texts = [_fmt(m) for m in examples]
texts = [t for t in texts if len(tokenizer.encode(t)) <= MAX_SEQ]
print(f'Training examples after length filter: {len(texts):,}')

dataset   = Dataset.from_dict({'text': texts})
train_val = dataset.train_test_split(test_size=0.05, seed=42)
print(f'Train {len(train_val["train"]):,}  |  Val {len(train_val["test"]):,}')

# ── Training config ───────────────────────────────────────────────────────────
# For a ~971 MB model, 3 epochs trains in ~5-15 min on T4
EPOCHS = 3 if not USE_4BIT else 2
training_args = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2 if not USE_4BIT else 1,
    gradient_accumulation_steps=4 if not USE_4BIT else 8,
    learning_rate=2e-4 if not USE_4BIT else 1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    max_grad_norm=0.3,
    optim='adamw_torch' if not USE_4BIT else 'paged_adamw_8bit',
    fp16=False, bf16=True,
    logging_steps=10,
    eval_strategy='steps', eval_steps=50,
    save_strategy='steps', save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    max_seq_length=MAX_SEQ,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_val['train'],
    eval_dataset=train_val['test'],
    tokenizer=tokenizer,
)

print(f'\nStarting fine-tune ({EPOCHS} epochs, LoRA r={LORA_R}) ...')
import time; t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✓ Done in {elapsed/60:.1f} min')
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'✓ Adapter → {ADAPTER_DIR}')
print('\n✓ Cell 4 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Merge LoRA adapter → save FP16 checkpoint  (~1–2 min for ~1 GB model)
# ══════════════════════════════════════════════════════════════════════════════
import gc, os, torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

ADAPTER_DIR = '/content/gemma4_31b_adapter'
MERGED_DIR  = '/content/gemma4_31b_axiom_merged'
MODEL_DIR   = Path('/content/gemma4_31b')
HF_TOKEN    = os.environ.get('HF_TOKEN', '')
MODEL_ID    = os.environ.get('SRD_MODEL_ID', 'google/gemma-4-31B-it-assistant')

assert Path(ADAPTER_DIR).is_dir(), 'Adapter not found — run Cell 4 first'

# Free training model
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Freed training model')

# Reload base in BF16 for merge
import time; t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto',
    token=HF_TOKEN or None, cache_dir=str(MODEL_DIR))
tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print(f'Base reloaded in {(time.time()-t0):.1f}s')

merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
print('✓ LoRA merged')

merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tok.save_pretrained(MERGED_DIR)
size_mb = sum(f.stat().st_size for f in Path(MERGED_DIR).rglob('*.safetensors')) / 1024**2
print(f'✓ Saved → {MERGED_DIR}  ({size_mb:.0f} MB)')

del merged, base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\n✓ Cell 5 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — SRD-4 pack merged model → gemma4_31b_srd4.axm
#           ~971 MB input → ~275 MB .axm  (~3–8 min)
# ══════════════════════════════════════════════════════════════════════════════
import json, os, subprocess, sys, time
from pathlib import Path

REPO     = Path('/content/axiom')
MERGED   = Path('/content/gemma4_31b_axiom_merged')
AXM_PATH = Path('/content/gemma4_31b_srd4.axm')
RESULTS  = REPO / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

assert MERGED.is_dir(), 'Merged model not found — run Cell 5 first'
assert os.environ.get('AXIOM_MASTER_KEY'), 'AXIOM_MASTER_KEY missing — re-run Cell 1'

print('Packing → SRD-4 .axm ...')
print(f'  Input  : {MERGED}  ({sum(f.stat().st_size for f in MERGED.rglob("*.safetensors"))/1024**2:.0f} MB)')
print(f'  Output : {AXM_PATH}')
print(f'  Est.   : ~275 MB  (~4.5 bpw)')

t0 = time.time()
subprocess.run(
    [sys.executable, 'axm_cli.py', 'pack',
     '--model',      str(MERGED),
     '--srd4', '--real-pack',
     '--output',     str(AXM_PATH),
     '--stats-json', str(RESULTS / 'gemma4_31b_pack.json')],
    cwd=REPO, check=True,
)
elapsed = time.time() - t0
size_mb = AXM_PATH.stat().st_size / 1024**2
try:
    stats = json.loads((RESULTS/'gemma4_31b_pack.json').read_text())
except Exception:
    stats = {}
print(f'\n✓ Packed in {elapsed/60:.1f} min')
print(f'  .axm     : {size_mb:.0f} MB')
print(f'  bpw      : {stats.get("quant",{}).get("bpw","N/A")}')
print(f'  fingerprint: {stats.get("fingerprint","N/A")}')
print('\n✓ Cell 6 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Verify .axm proof ledger  (~5 s)
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys
from pathlib import Path

REPO     = Path('/content/axiom')
AXM_PATH = Path('/content/gemma4_31b_srd4.axm')
assert AXM_PATH.is_file(), '.axm not found — run Cell 6 first'

result = subprocess.run(
    [sys.executable, 'axm_cli.py', 'verify', str(AXM_PATH)],
    cwd=REPO, capture_output=True, text=True)
try:
    output = json.loads(result.stdout)
except Exception:
    output = {'verified': False, 'raw': result.stdout + result.stderr}
print(json.dumps(output, indent=2))
if not output.get('verified'):
    raise RuntimeError('✗ Verification FAILED')
FINGERPRINT = output.get('fingerprint', '')
print(f'\n✓ Verified  fingerprint={FINGERPRINT}')
print('\n✓ Cell 7 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Extract .axm → FP16 → GGUF Q4_K_M  (~5–10 min)
#           ~275 MB .axm → ~295 MB GGUF
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, time
from pathlib import Path

REPO      = Path('/content/axiom')
AXM_PATH  = Path('/content/gemma4_31b_srd4.axm')
GGUF_PATH = Path('/content/gemma4_31b_q4km.gguf')
LLAMA_DIR = Path('/content/llama.cpp')
assert AXM_PATH.is_file(), '.axm not found — run Cell 6 first'
assert (LLAMA_DIR/'build/bin/llama-quantize').is_file(), 'llama-quantize missing — re-run Cell 1'

device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f'Extracting → GGUF Q4_K_M on {device} ...')
t0 = time.time()
subprocess.run(
    [sys.executable, '-m', 'research.quant.axm_to_gguf',
     '--container', str(AXM_PATH),
     '--gguf-out',  str(GGUF_PATH),
     '--llamacpp',  str(LLAMA_DIR),
     '--quant',     'Q4_K_M',
     '--device',    device],
    cwd=REPO, check=True,
)
elapsed = time.time() - t0
size_mb = GGUF_PATH.stat().st_size / 1024**2
print(f'\n✓ {elapsed/60:.1f} min  →  {size_mb:.0f} MB  ({GGUF_PATH.name})')
print('\n✓ Cell 8 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Axiom MET metadata sidecar
#           Reads actual config.json so all numbers are real, not guesses.
# ══════════════════════════════════════════════════════════════════════════════
import json, time
from pathlib import Path

GGUF_PATH = Path('/content/gemma4_31b_q4km.gguf')
MERGED    = Path('/content/gemma4_31b_axiom_merged')
assert GGUF_PATH.is_file(), 'GGUF not found — run Cell 8 first'

try:
    FP = FINGERPRINT
except NameError:
    FP = ''

# Read real architecture
cfg = json.loads((MERGED/'config.json').read_text()) if (MERGED/'config.json').is_file() else {}
HIDDEN  = cfg.get('hidden_size', 2048)
LAYERS  = cfg.get('num_hidden_layers', 18)
VOCAB   = cfg.get('vocab_size', 262144)
PARAMS  = sum(p.numel() for p in __import__('itertools').chain()) if False else (
    LAYERS * HIDDEN * 12 + VOCAB * HIDDEN * 2
)   # rough estimate; real value printed in Cell 3

EMB_MB = round(VOCAB * HIDDEN * 2 / 1024**2, 1)   # F16
GGUF_MB = round(GGUF_PATH.stat().st_size / 1024**2)

# Split layers into 4 MET slots  (proportions: early 20% / factual 20% / reasoning 37% / governance 23%)
def _split(n, fracs):
    cuts, lo = [], 0
    for f in fracs:
        hi = min(lo + max(1, round(n * f)), n)
        cuts.append((lo, hi - 1))
        lo = hi
    cuts[-1] = (cuts[-1][0], n - 1)
    return cuts

RNGS = _split(LAYERS, [0.20, 0.20, 0.37, 0.23])
SLOTS = ['early','factual','reasoning','governance']
TRANS_MB = GGUF_MB - EMB_MB   # transformer portion of GGUF
SLOT_RANGES = {}
for nm, (lo, hi), frac in zip(SLOTS, RNGS, [0.20, 0.20, 0.37, 0.23]):
    SLOT_RANGES[nm] = {
        'layers': f'{lo}-{hi}', 'first_layer': lo, 'last_layer': hi,
        'mb': round(max(1, TRANS_MB * frac)), 'precision': 'Q4_K_M'
    }

HYDRATION = {
    'INFORM':    ['early'],
    'CLARIFY':   ['early','governance'],
    'REFUSE':    ['early','governance'],
    'UNCERTAIN': ['early','governance'],
    'HARM':      ['early','factual','reasoning','governance'],
    'DECEIVE':   ['early','factual','reasoning','governance'],
}

chunk_map = {}
for s, info in SLOT_RANGES.items():
    for i in range(info['first_layer'], info['last_layer']+1):
        chunk_map[str(i)] = s

slot_mb = {s: SLOT_RANGES[s]['mb'] for s in SLOTS}
# Small model (< 1 GB GGUF) → phone-class storage speed
STORAGE_MBS = 1500 if GGUF_MB < 1500 else 3000
intent_ram = {}
for intent, chunks in HYDRATION.items():
    xmb = sum(slot_mb[c] for c in chunks)
    intent_ram[intent] = {
        'chunks': chunks, 'transformer_mb': xmb,
        'total_mb': EMB_MB + xmb,
        'load_ms': round(xmb / STORAGE_MBS * 1000, 1),
    }

meta = {
    'axiom_version': '1.4',
    'generated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'model_id': 'google/gemma-4-31B-it-assistant (Axiom fine-tune)',
    'gguf_path': str(GGUF_PATH),
    'gguf_mb': GGUF_MB,
    'fingerprint': FP,
    'architecture': {'hidden_size': HIDDEN, 'num_layers': LAYERS, 'vocab_size': VOCAB},
    'fine_tuned_on': 'Axiom training corpus (behavioral + metric-targeted)',
    'embedding_slot': {
        'mb': EMB_MB, 'precision': 'F16', 'always_pinned': True,
        'note': f'{VOCAB:,}-token vocab  {EMB_MB:.0f} MB F16'
    },
    'transformer_chunks': SLOT_RANGES,
    'chunk_map': chunk_map,
    'hydration_policy': HYDRATION,
    'intent_ram_budget': intent_ram,
    'storage_speed_mbs': STORAGE_MBS,
    'between_met_floor_mb': EMB_MB,
    'peak_harm_mb': EMB_MB + sum(slot_mb.values()),
}

sidecar = GGUF_PATH.with_suffix('.axiom_meta.json')
sidecar.write_text(json.dumps(meta, indent=2) + '\n')
print(f'✓ {sidecar}')
print()
print(f'  Model    : {LAYERS} layers · hidden {HIDDEN} · vocab {VOCAB:,}')
print(f'  GGUF     : {GGUF_MB} MB')
print(f'  Embedding: {EMB_MB:.0f} MB F16  (always pinned)')
print()
print(f'  {"Slot":<14}  {"Layers":<8}  {"MB":>5}')
print('  ' + '─'*34)
print(f'  {"embedding":<14}  {"—embed—":<8}  {EMB_MB:>5.0f}  ✓ pinned')
for s, info in SLOT_RANGES.items():
    print(f'  {s:<14}  {info["layers"]:<8}  {info["mb"]:>5}')
print()
print(f'  {"Intent":<10}  {"Total MB":>9}  {"Load ms":>8}')
print('  ' + '─'*32)
for intent, info in intent_ram.items():
    print(f'  {intent:<10}  {info["total_mb"]:>9.0f}  {info["load_ms"]:>6.1f}ms')
print('\n✓ Cell 9 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 10 — Smoke test + memory dashboard + download
#            ~295 MB GGUF → small enough for files.download directly
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, time
from pathlib import Path

GGUF_PATH = Path('/content/gemma4_31b_q4km.gguf')
LLAMA_CLI = Path('/content/llama.cpp/build/bin/llama-cli')
sidecar   = GGUF_PATH.with_suffix('.axiom_meta.json')
assert GGUF_PATH.is_file(), 'GGUF not found — run Cell 8 first'
assert LLAMA_CLI.is_file(), 'llama-cli not found — re-run Cell 1'

# ── Smoke test ────────────────────────────────────────────────────────────────
PROMPT = ('<start_of_turn>user\n'
          'Classify using Axiom verdict schema: '
          '"Delete all user records immediately."<end_of_turn>\n'
          '<start_of_turn>model\n')
print('Smoke test (32 tokens) ...')
t0 = time.time()
ngl = '99' if __import__('torch').cuda.is_available() else '0'
result = subprocess.run(
    [str(LLAMA_CLI),'-m',str(GGUF_PATH),'-p',PROMPT,
     '-n','32','--temp','0.0',f'-ngl',ngl,'--log-disable'],
    capture_output=True, text=True, timeout=120)
if result.returncode != 0:
    print(f'✗ failed: {result.stderr[-300:]}')
else:
    print(f'  Output : {repr(result.stdout.strip()[-140:])}')
    print(f'  Time   : {(time.time()-t0):.1f}s')
    print('  ✓ GGUF loads and generates')

# ── Memory dashboard ──────────────────────────────────────────────────────────
if sidecar.is_file():
    meta   = json.loads(sidecar.read_text())
    EMB_MB = meta['embedding_slot']['mb']
    PEAK   = meta['peak_harm_mb']
    GGUF_MB= meta['gguf_mb']
    ir     = meta.get('intent_ram_budget', {})
    W = 30
    def bar(mb): n=round(min(mb/max(PEAK,1),1)*W); return '█'*n+'░'*(W-n)
    print()
    print('═'*68)
    print('  AXIOM MET HYDRATION — Gemma 4 31B-it-assistant (Axiom fine-tuned)')
    print('─'*68)
    arch = meta.get('architecture', {})
    print(f'  {arch.get("num_layers","?")} layers · hidden {arch.get("hidden_size","?")} · '
          f'vocab {arch.get("vocab_size",0):,}')
    print(f'  GGUF : {GGUF_MB} MB  |  Fingerprint: {meta.get("fingerprint") or "(not set)"}')
    print()
    print(f'  {"Static GGUF (full in RAM)":<34}  {GGUF_MB:>5} MB  {bar(GGUF_MB)}')
    print(f'  {"Embedding floor (between METs)":<34}  {EMB_MB:>5.0f} MB  {bar(EMB_MB)}')
    print(f'  {"Peak HARM (all chunks)":<34}  {PEAK:>5.0f} MB  {bar(PEAK)}')
    print()
    avg = (0.70*ir.get('INFORM',{}).get('total_mb',0)
          +0.20*ir.get('CLARIFY',{}).get('total_mb',0)
          +0.09*ir.get('REFUSE',{}).get('total_mb',0)
          +0.01*ir.get('HARM',{}).get('total_mb',0))
    for intent, info in ir.items():
        marker = '◄' if intent in ('HARM','DECEIVE') else ''
        print(f'  {intent:<10}  {info["total_mb"]:>5.0f} MB  {bar(info["total_mb"])}  {marker}')
    print()
    print(f'  Avg RAM (70/20/9/1 workload): {avg:.0f} MB')
    storage_lbl = 'UFS 3.1' if meta.get('storage_speed_mbs',0) == 1500 else 'NVMe'
    print(f'  Storage tier: {storage_lbl} ({meta.get("storage_speed_mbs")} MB/s)')
    print('═'*68)

print('\n✓ Pipeline complete.')
print(f'  GGUF    : {GGUF_PATH}  ({GGUF_PATH.stat().st_size/1024**2:.0f} MB)')
print(f'  Sidecar : {sidecar}')
print()
print('Download (small enough for direct download):')
print('  from google.colab import files')
print(f'  files.download("{GGUF_PATH}")')
print(f'  files.download("{sidecar}")')
print()
print('Phone deploy (adb):')
print(f'  adb push gemma4_31b_q4km.gguf /storage/emulated/0/models/')
print(f'  adb push gemma4_31b_q4km.axiom_meta.json /storage/emulated/0/models/')